# Compute daily mean wind speed

To align with DCPP model data, we will compute daily wind speed from daily mean u and v components.

- ERA5
- BARRA-R2
- BARRA-C2 later - not a reanalysis as no DA

In [1]:
from dask_setup import setup_dask_client 

from pathlib import Path
import glob
from zarr.codecs import BloscCodec

import xarray as xr
import numpy as np

In [2]:
client, cluster, dask_tmp = setup_dask_client(mode="interactive", workload_type="cpu")   # heavy compute

INFO     [client] Starting Dask client setup (workload_type=cpu | environment=jupyter)
INFO     [resources] Resources detected via PBS (total_cores=104 | total_mem_gib=496.0)


INFO     [client] Temp/spill dir: /jobfs/181555512.gadi-pbs/dask-392145
INFO     [client] Workers: 104 | threads/worker: 1 | processes: True
INFO     [client] Mem: total ~496.0 GiB | usable ~446.0 GiB | per-worker ~4.3 GiB
INFO     [client] Compression: spill=auto | comm=False
INFO     [client] Dask client ready


[setup_dask_client] Configuration summary
temp/spill dir: /jobfs/181555512.gadi-pbs/dask-392145
Workers: 104 | threads/worker: 1 | processes: True
Memory: total ~496.0 GiB | usable ~446.0 GiB | per-worker ~4.3 GiB
Compression: spill=auto | comm=False


2026-10-19 15:08:47,686 - distributed.scheduler - ERROR - Task ('rechunk-merge-rechunk-transfer-9bab60dd57599e39209def7a5b4ee724', 0, 0, 2, 5, 0, 3) has 6.92 GiB worth of input dependencies, but worker tcp://127.0.0.1:43221 has memory_limit set to 4.29 GiB.
2026-10-19 15:14:18,971 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 12f46f5f951bab6d4080b4017d5376e0 initialized by task ('rechunk-merge-rechunk-transfer-e5f1c857d683a844581ad4af6ad73ff1', 0, 0, 2, 99, 1, 3) executed on worker tcp://127.0.0.1:40859
2026-10-19 15:14:19,339 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f6c923b7edca2e22f7fdf8459c6229e0 initialized by task ('rechunk-merge-rechunk-transfer-e5f1c857d683a844581ad4af6ad73ff1', 0, 1, 1, 99, 2, 2) executed on worker tcp://127.0.0.1:37617
2026-10-19 15:14:19,646 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle cb72fcf306f8394da26bd8a15a4bfe23 initialized by task ('rechunk-merge-rechunk-transfer-e5f1c857d683a844581ad4af6ad73ff1', 0, 1, 2

In [3]:
%cd /g/data/w42/dr6273/work/wind_drought/
import src.regions as regions
import src.wind as wind

%load_ext autoreload
%autoreload 2

/g/data/w42/dr6273/work/wind_drought


# Compute wind speed by year

In [4]:
def era_preprocess(ds):
    """
    Preprocess function for open_mfdataset.
    Selects Australian region, computes daily mean and renames coords.
    """
    ds = ds.sel(
        longitude=slice(REGION[0], REGION[1]),
        latitude=slice(REGION[2], REGION[3])
    )
    ds = ds.rename(
        {'longitude': 'lon',
         'latitude': 'lat'}
    )
    # ds = ds.coarsen(time=24).mean()
    # ds["time"] = ds["time"].dt.floor("D")
    return ds

In [5]:
def get_files(path, var, start_year=1979, end_year=2025):
    """Return files for years from start_year to end_year, inclusive."""
    files = (
        Path(path) / var
    ).glob("*/*.nc")

    return sorted(
        f for f in files
        if f.parent.name.isdigit()
        and start_year <= int(f.parent.name) <= end_year
    )

In [6]:
def open_era(files):
    """
    Load and preprocess hourly data for a given year
    """
    # Open all hours in the year (~33 GB)
    ds = xr.open_mfdataset(
        files,
        preprocess=era_preprocess,
        chunks='auto',
        parallel=True
    )
    return ds

In [7]:
def specify_encoding(ds):
    codec = BloscCodec(cname="zstd", clevel=6, shuffle="bitshuffle")

    encoding = {}
    for name, da in ds.data_vars.items():
        chunks = tuple(da.chunksizes[dim][0] for dim in da.dims)
        encoding[name] = {
            "chunks": chunks,
            "compressors": codec,
            "dtype": "float64",
        }

    return encoding

In [8]:
def write_zarr(ds, path, filename, encoding):
    """ Write to zarr """
    ds.to_zarr(
        path + filename,
        mode='w',
        encoding=encoding,
        zarr_format=3,
        consolidated=False
    )

In [9]:
ERA_PATH = "/g/data/rt52/era5/single-levels/reanalysis/" # hourly instantaneous

WRITE_PATH = "/g/data/ng72/dr6273/work/projects/wind_drought/data/ERA5/"

In [10]:
REGION = regions.get_Aus_boundary()

In [11]:
YEARS = range(1979, 2026)

100m u and v

In [12]:
u100_files = get_files(ERA_PATH, '100u', start_year=YEARS[0], end_year=YEARS[-1])
v100_files = get_files(ERA_PATH, '100v', start_year=YEARS[0], end_year=YEARS[-1])

In [13]:
len(u100_files)

564

In [14]:
u100 = open_era(u100_files)
v100 = open_era(v100_files)

In [15]:
u100

<xarray.Dataset> Size: 84GB
Dimensions:  (time: 412008, lat: 141, lon: 181)
Coordinates:
  * time     (time) datetime64[ns] 3MB 1979-01-01 ... 2025-12-31T23:00:00
  * lat      (lat) float32 564B -10.0 -10.25 -10.5 -10.75 ... -44.5 -44.75 -45.0
  * lon      (lon) float32 724B 110.0 110.2 110.5 110.8 ... 154.5 154.8 155.0
Data variables:
    u100     (time, lat, lon) float64 84GB dask.array<chunksize=(186, 12, 76), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2020-10-04 20:24:45 UTC+1100 by era5_replication_tools-1.2....
    license:      Licence to use Copernicus Products: https://apps.ecmwf.int/...
    summary:      ERA5 is the fifth generation ECMWF atmospheric reanalysis o...
    title:        ERA5 single-levels reanalysis 100m_u_component_of_wind 1979...

In [16]:
# u100['u100'].coarsen(time=24).mean().chunk({"time": -1, "lat": -1, "lon": 16})

In [35]:
u100_chunked = u100.chunk({"time": -1, "lat": 25, "lon": 3})
v100_chunked = v100.chunk({"time": -1, "lat": 25, "lon": 3})

In [36]:
u100_chunked = u100_chunked["u100"]
v100_chunked = v100_chunked["v100"]

In [37]:
v100_chunked

<xarray.DataArray 'v100' (time: 412008, lat: 141, lon: 181)> Size: 84GB
dask.array<rechunk-p2p, shape=(412008, 141, 181), dtype=float64, chunksize=(412008, 25, 3), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 3MB 1979-01-01 ... 2025-12-31T23:00:00
  * lat      (lat) float32 564B -10.0 -10.25 -10.5 -10.75 ... -44.5 -44.75 -45.0
  * lon      (lon) float32 724B 110.0 110.2 110.5 110.8 ... 154.5 154.8 155.0
Attributes:
    units:      m s**-1
    long_name:  100 metre V wind component

In [38]:
wind_speed = np.sqrt(u100_chunked ** 2 + v100_chunked ** 2)

In [27]:
# wind_speed = wind_speed.chunk({"time": -1, "lat": -1, "lon": 16})

In [49]:
capacity_factor = wind.capacity_factor(wind_speed)

In [50]:
capacity_factor = capacity_factor.coarsen(time=24).mean()
capacity_factor["time"] = capacity_factor["time"].dt.floor("D")
capacity_factor = capacity_factor.chunk({"time": -1, "lat": -1, "lon": 16})

In [51]:
capacity_factor

<xarray.DataArray (time: 17167, lat: 141, lon: 181)> Size: 4GB
dask.array<rechunk-merge, shape=(17167, 141, 181), dtype=float64, chunksize=(17167, 141, 16), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 137kB 1979-01-01 1979-01-02 ... 2025-12-31
  * lat      (lat) float32 564B -10.0 -10.25 -10.5 -10.75 ... -44.5 -44.75 -45.0
  * lon      (lon) float32 724B 110.0 110.2 110.5 110.8 ... 154.5 154.8 155.0
Attributes:
    units:    m s**-1

In [52]:
capacity_factor.isel(time=0).plot()

KeyboardInterrupt: 

In [67]:
# wind_dir = np.mod(180 + (180 / np.pi) * np.arctan2(u100_chunked, v100_chunked), 360)
# wind_dir = wind_dir.rename("wind_direction").drop_attrs()

In [28]:
wind_speed = wind_speed.chunk({"time": -1, "lat": -1, "lon": 16})
# wind_dir = wind_dir.chunk({"time": -1, "lat": -1, "lon": 5})

In [39]:
wind_speed

<xarray.DataArray (time: 412008, lat: 141, lon: 181)> Size: 84GB
dask.array<sqrt, shape=(412008, 141, 181), dtype=float64, chunksize=(412008, 25, 3), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 3MB 1979-01-01 ... 2025-12-31T23:00:00
  * lat      (lat) float32 564B -10.0 -10.25 -10.5 -10.75 ... -44.5 -44.75 -45.0
  * lon      (lon) float32 724B 110.0 110.2 110.5 110.8 ... 154.5 154.8 155.0
Attributes:
    units:    m s**-1

In [70]:
wind = wind_speed.to_dataset(name="wind_speed")

In [46]:
# capacity_factor = capacity_factor.to_dataset(name="capacity_factor")

capacity_factor_encoding = specify_encoding(capacity_factor)

In [72]:
wind

<xarray.Dataset> Size: 4GB
Dimensions:     (lon: 181, lat: 141, time: 17167)
Coordinates:
  * lon         (lon) float32 724B 110.0 110.2 110.5 110.8 ... 154.5 154.8 155.0
  * lat         (lat) float32 564B -10.0 -10.25 -10.5 ... -44.5 -44.75 -45.0
  * time        (time) datetime64[ns] 137kB 1979-01-01 1979-01-02 ... 2025-12-31
Data variables:
    wind_speed  (time, lat, lon) float64 4GB dask.array<chunksize=(17167, 141, 16), meta=np.ndarray>

In [73]:
print(wind)

<xarray.Dataset> Size: 4GB
Dimensions:     (lon: 181, lat: 141, time: 17167)
Coordinates:
  * lon         (lon) float32 724B 110.0 110.2 110.5 110.8 ... 154.5 154.8 155.0
  * lat         (lat) float32 564B -10.0 -10.25 -10.5 ... -44.5 -44.75 -45.0
  * time        (time) datetime64[ns] 137kB 1979-01-01 1979-01-02 ... 2025-12-31
Data variables:
    wind_speed  (time, lat, lon) float64 4GB dask.array<chunksize=(17167, 141, 16), meta=np.ndarray>


In [79]:
wind_encoding = specify_encoding(wind)
wind_encoding

{'wind_speed': {'chunks': (17167, 141, 16),
  'compressors': BloscCodec(_tunable_attrs={'typesize'}, typesize=1, cname=<BloscCname.zstd: 'zstd'>, clevel=6, shuffle=<BloscShuffle.bitshuffle: 'bitshuffle'>, blocksize=0),
  'dtype': 'float64'}}

In [47]:
write_zarr(
    capacity_factor,
    WRITE_PATH,
    'cf_test.zarr',
    capacity_factor_encoding
)

/jobfs/181555512.gadi-pbs/ipykernel_392145/2978423523.py:3: UserWarning: Times can't be serialized faithfully to int64 with requested units 'hours since 1900-01-01'. Serializing with units 'minutes since 1900-01-01' instead. Set encoding['dtype'] to floating point dtype to serialize with units 'hours since 1900-01-01'. Set encoding['units'] to 'minutes since 1900-01-01' to silence this warning .
  ds.to_zarr(


KeyboardInterrupt: 

2026-10-19 15:28:32,577 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/worker.py", line 1273, in heartbeat
    response = await retry_operation(
               ^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/utils_comm.py", line 416, in retry_operation
    return await retry(
           ^^^^^^^^^^^^
  File "/g/d

In [13]:
def era_windspeed(u_name, v_name, ws_name, years, write_path):
    """
    Compute wind speed from daily mean u and v winds.

    u_name, v_name: str, names of u and v variables in ERA5
    ws_name: str, desired wind speed variable name
    years: range, years to process
    write_path: str, path to write directory
    """
    da_list = []
    for year in years:
        if year % 5 == 0:
            print(year)
            
        u = load_hourly_era5(
            era_preprocess,
            u_name,
            year
        )
    
        v = load_hourly_era5(
            era_preprocess,
            v_name,
            year
        )
        
        ws = fn.windspeed(
            u.rename({"u"+u_name[:-1]: ws_name}),
            v.rename({"v"+v_name[:-1]: ws_name})
        )
        # ws = ws.chunk({"time": -1})
        
        # encoding = {
        #     ws_name: {"dtype": "float32"}
        # }
        # w100.to_netcdf(
        #     write_path + "100w_era5_hourly_" + str(year) + "_Aus.nc",
        #     mode="w",
        #     encoding=encoding
        # )
        da_list.append(ws)
        
    ws = xr.concat(da_list, dim="time")
    ws = ws.chunk({"time": "300MB"})
        
    encoding = {
        ws_name: {"dtype": "float32"}
    }
    ws.to_zarr(
        write_path + ws_name + "_era5_daily_Aus.zarr",
        consolidated=True,
        mode="w",
        zarr_format=2, # Need this to prevent an error with zarr3: https://github.com/pydata/xarray/issues/10032
        encoding=encoding
    )

In [14]:
# era_windspeed("100u", "100v", "ws100m", YEARS, WRITE_PATH)

In [15]:
# SFC_YEARS = range(1940, 2025)

In [16]:
era_windspeed("10u", "10v", "ws10m", YEARS, WRITE_PATH)

1940
1945
1950
1955
1960
1965
1970
1975
1980
1985
1990
1995
2000
2005
2010
2015
2020


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 54.69 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


# Close cluster

In [17]:
client.close()
cluster.close()

INFO:dask_jobqueue.pbs:Resource specification for PBS not set, initializing it to select=1:ncpus=48:mem=179GB
